# Random Train/Val Split Analysis (25 States)

This notebook is the cleaned final version of the 25-state random train/val split analysis.
It regenerates the saved outputs used downstream in `ahmm_final_figures.ipynb`.


In [ ]:
import os

import pandas as pd
import scipy.io as sio

from ahmm_eval import (
    build_df_all_for_violin_both,
    build_optimal_height_distance_matrix,
    load_neural_covariances,
    model_eval_behavior_cross_compare,
    model_eval_behavior_plot,
    model_eval_ks_search,
    model_eval_pipeline,
    model_eval_pipeline_plot,
    plot_neural_and_top10_models,
    plot_violin_with_pbars,
    save_violin_reps_to_mat_stacked,
)
from ahmm_utils import SingleTrackConfig, SingleTrackGenerator, build_vocab



In [ ]:
os.makedirs("Paper/Figures/Data", exist_ok=True)
os.makedirs("Data/Model_behavior/25", exist_ok=True)

sessions_combined = pd.read_pickle(resolve_data_path("sessions_combined.pkl"))

heights = sorted({
    int(h) for sess in sessions_combined for (h, _, _) in sess["records"]
} | {
    int(h) for sess in sessions_combined for (_, h, _) in sess["records"]
})

vocab = build_vocab(heights, height_encoding="split")
cfg = SingleTrackConfig(
    tower_heights=heights,
    height_encoding="split",
    p_gap=1.0,
    p_reward_given_correct=1.0,
    p_reward_given_incorrect=0.0,
)
gen = SingleTrackGenerator(vocab, cfg)



## Path Fixup Note

`OLD_ROOT` and `NEW_ROOT` are only for repairing legacy `save_path` entries in `model_log.pkl` when the files were moved after training.

Example from the older workflow:
- `OLD_ROOT = "ahmm_models_per_session_all/"`
- `NEW_ROOT = "ahmm_models_per_session_all/random_train_val_split/"`

If your logged paths already match the current filesystem, you can skip this rewrite.


In [ ]:
OLD_ROOT = "ahmm_models_per_session_all/"
NEW_ROOT = "ahmm_models_per_session_all/random_train_val_split/"

def fix_model_path(path):
    path = str(path).replace("\\", "/")
    if path.startswith(OLD_ROOT):
        path = path.replace(OLD_ROOT, NEW_ROOT, 1)
    return os.path.normpath(path)



In [ ]:
optimal_mat = build_optimal_height_distance_matrix(
    h_min=200,
    h_max=650,
    step=50,
    as_dataframe=False,
)[0]

all_neural = load_neural_covariances("neural_covariances.mat")



## 25-State Random Split Pipeline


In [ ]:
df_25_rand, sim_df_25_rand, sim_df_25_rand_filtered = model_eval_pipeline(
    "ahmm_models_per_session_all/random_train_val_split/25/model_log.pkl",
    all_neural,
)

model_eval_pipeline_plot(sim_df_25_rand_filtered, 25)
model_eval_ks_search(sim_df_25_rand_filtered, 25, k1=30, k2=80)



In [ ]:
df_all_violin_rand_25, dfv_rand_25 = build_df_all_for_violin_both(
    sim_df_25_rand_filtered,
    k1=1,
    k2=10,
    optimal_mat=optimal_mat,
    all_neural=all_neural,
    n=1,
    ascending=True,
)
plot_violin_with_pbars(df_all_violin_rand_25, "rank", ascending=True)

save_violin_reps_to_mat_stacked(
    sim_df_25_rand_filtered,
    k1=1,
    k2=10,
    n=1,
    optimal_mat=optimal_mat,
    all_neural=all_neural,
    out_path="Paper/Figures/Data/selected_models_k1_1_k2_10_n1.mat",
    compare_metric="pearson",
    on_constant="nan",
    ascending=True,
)

sio.savemat(
    "Paper/Figures/Data/data_4o_model_neural_covariance_similarity.mat",
    {
        "Similarity": df_all_violin_rand_25["similarity"].values,
        "Rank": df_all_violin_rand_25["rank"].values,
        "Group": df_all_violin_rand_25["group"].values,
    },
)

df_25_rand["save_path"] = df_25_rand["save_path"].apply(fix_model_path)
df_25_rand.to_pickle("Paper/Figures/Data/df_25_rand.pkl")
dfv_rand_25.to_pickle("Paper/Figures/Data/dfv_rand_25.pkl")
sim_df_25_rand_filtered.to_pickle("Paper/Figures/Data/sim_df_25_rand.pkl")



In [ ]:
figs = plot_neural_and_top10_models(
    all_neural=all_neural,
    df_models=df_25_rand,
    df_used=dfv_rand_25,
    within_group_label="AA",
    topk=3,
    rows_per_fig=80,
)

cg_df_rand_25 = model_eval_behavior_cross_compare(
    df_25_rand,
    sessions_combined,
    gen,
    vocab,
    test_ratio=0.3,
    valid_ratio=0,
    T_model=3000,
)
cg_df_rand_25.to_pickle(
    "Data/Model_behavior/25/rand_train_val_cg_rank1_pde_behavior_cross_compare_results_25_states.pkl"
)

df_grouped_rand_25 = model_eval_behavior_plot(
    cg_df_rand_25,
    metric="model_corr_heatmap_similarity_score",
    ascending=True,
)

